# Sensor placement optimization — MuJoCo (Colab, self-contained)

**Requires only this notebook + pip (and a Colab or local kernel).** No separate project folder on your machine. The
optimizer sources ship **inside the notebook** as a base64 zip; the **smoke YAML** is also embedded in base64
(`configs/mujoco_colab_smoke.yaml` is written for you at runtime).

1. **Install** — the full `requirements.txt` from the project (JAX, scikit-learn, MLflow, PyTorch, Matplotlib, pandas, rich, MuJoCo, CMA, …) is embedded at notebook generation time; `pip install -r` in a temp file (same style as the SysID notebook).

2. **Materialize** — default: **decode the embedded project zip** into `/content/sensor_placement_opt_local/`
   (set `USE_EMBEDDED_SOURCE_BUNDLE = False` in that cell to use GitHub download, Colab upload, or a local `cwd` that
   already contains `sensor_opt/` instead).

3. **Smoke config** — writes `mujoco_colab_smoke.yaml` from the embedded copy.

4. **MuJoCo sanity** — minimal MJCF + one `mj_step` (lightweight, SysID-style check).

5. **Run** — `python -m sensor_opt.run_experiment` (short CMA, short MuJoCo rollouts on CPU).

6. **Plots** — CMA-ES generations from `results/.../generations.csv`.

**Refresh the embedded zip:** from the repository root run
`python3 scripts/emit_colab_sensor_placement_notebook.py`
after changing `sensor_opt/` or `configs/`.


In [ ]:
# Pinned in repo `requirements.txt` — full stack (JAX, sklearn, MLflow, torch, matplotlib, …)
import os
import subprocess
import sys
import tempfile
from pathlib import Path

REQUIREMENTS_SENSOR_PLACEMENT_COLAB = "cma>=3.3,<4\nnumpy>=1.24,<2.0\nscipy>=1.10,<1.14\npyyaml>=6.0,<7\npandas>=2.1,<2.3\npytest>=8.0,<9\npytest-cov>=4.1,<6\nmlflow>=2.10,<3\nrich>=13,<14\njax>=0.4.30,<0.5\nscikit-learn>=1.3,<1.6\nmujoco>=3.1.0\nmatplotlib>=3.7,<4\ntorch>=2.0,<3"
fd, _req_path = tempfile.mkstemp(suffix="-pip-reqs.txt", text=True)
os.close(fd)
try:
    Path(_req_path).write_text(REQUIREMENTS_SENSOR_PLACEMENT_COLAB, encoding="utf-8")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "pip", "setuptools", "wheel"])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", _req_path])
finally:
    try:
        os.unlink(_req_path)
    except OSError:
        pass

import cma
import jax
import matplotlib
import mlflow
import mujoco
import numpy as np
import pandas as pd
import rich
import scipy
import sklearn  # scikit-learn
import torch
print("python:", sys.executable)
for name, mod in [
    ("cma", cma),
    ("jax", jax),
    ("matplotlib", matplotlib),
    ("mlflow", mlflow),
    ("mujoco", mujoco),
    ("numpy", np),
    ("pandas", pd),
    ("rich", rich),
    ("scipy", scipy),
    ("sklearn", sklearn),
    ("torch", torch),
]:
    v = getattr(mod, "__version__", "") or "?"
    if name == "mujoco":
        v = getattr(mujoco, "__version__", v) or "?"
    print(f"{name}: {v}")


In [ ]:
# --- Materialize: expand embedded `sensor_opt/` + `configs/` into WORK, or use fallbacks below ---
from __future__ import annotations

import base64
import io
import os
import shutil
import sys
import zipfile
from pathlib import Path
import urllib.request

# Colab uses /content; local Jupyter: use a folder under the current working directory.
if os.path.isdir("/content"):
    WORK = Path("/content/sensor_placement_opt_local")
else:
    WORK = (Path.cwd() / "_sensor_placement_opt_local").resolve()

# Set False to use download / Colab upload / current working tree instead of the bundle below.
USE_EMBEDDED_SOURCE_BUNDLE = True

# Embedded at notebook build time (see `scripts/emit_colab_sensor_placement_notebook.py`).
_PROJECT_ZIP_B64 = "..."

def _find_package_root(p: Path) -> Path | None:
    if (p / "sensor_opt" / "__init__.py").is_file():
        return p
    return None

def _search_tree_for_package(extract_dir: Path) -> Path | None:
    for init in extract_dir.rglob("sensor_opt/__init__.py"):
        return init.parent.parent
    return None

def _activate(project_root: Path) -> None:
    project_root = project_root.resolve()
    os.environ["SENSOR_PLACEMENT_NOTEBOOK_ROOT"] = str(project_root)
    os.chdir(project_root)
    s = str(project_root)
    if s not in sys.path:
        sys.path.insert(0, s)
    print("Using project root:", project_root)
    print("cwd:", os.getcwd())

def _extract_top_level_project_zip_to_work(data: bytes) -> None:
    if WORK.exists():
        shutil.rmtree(WORK)
    WORK.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(data), "r") as zf:
        zf.extractall(WORK)
    if not (WORK / "sensor_opt" / "__init__.py").is_file():
        raise FileNotFoundError("Project zip is missing sensor_opt/ at top level.")

def _extract_github_style_zip_to_work(data: bytes) -> None:
    WORK.parent.mkdir(parents=True, exist_ok=True)
    tmp = WORK.parent / f"{WORK.name}.tmp_extract"
    if tmp.exists():
        shutil.rmtree(tmp)
    tmp.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(data), "r") as zf:
        zf.extractall(tmp)
    found = _search_tree_for_package(tmp)
    if found is None:
        shutil.rmtree(tmp, ignore_errors=True)
        raise FileNotFoundError("ZIP did not contain sensor_opt/ — use the repository root (Download ZIP from GitHub).")
    if WORK.exists():
        shutil.rmtree(WORK)
    found.rename(WORK)
    shutil.rmtree(tmp, ignore_errors=True)

def materialize() -> Path:
    if USE_EMBEDDED_SOURCE_BUNDLE and _PROJECT_ZIP_B64:
        data = base64.b64decode(_PROJECT_ZIP_B64)
        _extract_top_level_project_zip_to_work(data)
        _activate(WORK)
        return WORK

    r0 = _find_package_root(WORK)
    if r0 is not None:
        _activate(r0)
        return r0

    env_root = (os.environ.get("SENSOR_PLACEMENT_EXTRACTED_ROOT") or "").strip()
    if env_root:
        p = Path(env_root).expanduser().resolve()
        r1 = _find_package_root(p)
        if r1 is None:
            raise FileNotFoundError(f"SENSOR_PLACEMENT_EXTRACTED_ROOT is not a project root: {p}")
        _activate(r1)
        return r1

    for start in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        r2 = _find_package_root(start)
        if r2 is not None:
            _activate(r2)
            return r2

    url = (os.environ.get("SENSOR_PLACEMENT_ZIP_URL") or "").strip()
    if url:
        print("Downloading:", url)
        req = urllib.request.Request(url, headers={"User-Agent": "sensor-placement-colab"})
        with urllib.request.urlopen(req, timeout=120) as resp:
            data = resp.read()
        _extract_github_style_zip_to_work(data)
        _activate(WORK)
        return WORK

    try:
        from google.colab import files
    except Exception:
        files = None
    if files is not None:
        print("Upload a ZIP of the project root. The archive must contain sensor_opt/.")
        up = files.upload()
        if not up:
            raise SystemExit("No file uploaded.")
        data = next(iter(up.values()))
        _extract_github_style_zip_to_work(data)
        _activate(WORK)
        return WORK

    raise RuntimeError(
        "Set USE_EMBEDDED_SOURCE_BUNDLE = True, or SENSOR_PLACEMENT_ZIP_URL, SENSOR_PLACEMENT_EXTRACTED_ROOT, "
        "or run in Colab and upload a project ZIP, or start Jupyter with cwd at the project root."
    )

PROJECT_ROOT = materialize()


In [ ]:
# --- Write bundled smoke config (base64) + verify import ---
import base64
import importlib
from pathlib import Path

_MUJOCO_LOCAL_SMOKE_YAML_B64 = "IyBGYXN0IGxvY2FsIENQVSB0ZXN0OiBzYW1lIGFzIG11am9jb19wcmlzbSB3aXRoIHRpbnkgQ01BICsgc2hvcnQgcm9sbG91dHMgKGZvciBub3RlYm9va3MgLyBDSSBzbW9rZSkuCiMgSW5zdGFsbCBQeXRob24gZGVwcyB3aXRoIGBwaXBgIChzZWUgYHJlcXVpcmVtZW50cy50eHRgIG9yIHRoZSBzZW5zb3ItcGxhY2VtZW50IG5vdGVib29rIGluc3RhbGwgY2VsbCkuCmV4cGVyaW1lbnQ6CiAgbmFtZTogIm11am9jb19sb2NhbF9zbW9rZSIKICBzZWVkOiA0MgoKc2VhcmNoOgogIHR5cGU6IGNtYQoKZml4ZWRfbW91bnRfb3JkZXI6IHRydWUKZml4ZWRfc2Vuc29yX2dlb21ldHJ5OiB0cnVlCgpkZWZhdWx0X3NlbnNvcl9wb3NlOgogIGFsbDoKICAgIHhfb2Zmc2V0OiAwLjAKICAgIHlfb2Zmc2V0OiAwLjAKICAgIHpfb2Zmc2V0OiAwLjAKICAgIHlhd19kZWc6IDAuMAogICAgcGl0Y2hfZGVnOiAwLjAKICAgIHJhbmdlX2ZyYWN0aW9uOiAxLjAKICAgIGhmb3ZfZnJhY3Rpb246IDEuMAoKc2Vuc29yX2J1ZGdldDoKICBsaWRhcjoKICAgIHVzZXJtYXg6IDIKICBjYW1lcmE6CiAgICB1c2VybWF4OiAyCiAgcmFkYXI6CiAgICB1c2VybWF4OiAyCgpzZW5zb3JfbW9kZWxzOgogIGxpZGFyOgogICAgaXNhYWM6IHsgcHJpbV9wYXRoOiAiIiB9CiAgICBtb2RlbF9uYW1lOiAiVmVsb2R5bmUgVkxQLTE2IgogICAgY29zdF91c2Q6IDQwMDAKICAgIHJhbmdlX206IDEwMC4wCiAgICBob3Jpem9udGFsX2Zvdl9kZWc6IDM2MC4wCiAgICB2ZXJ0aWNhbF9mb3ZfZGVnOiAzMC4wCiAgICBtYXNzX2tnOiAwLjgzCiAgICBjb21wdXRlX3RvcHM6IDguMAogICAgbWVtb3J5X2diOiAxLjIKICAgIGxhdGVuY3lfbXM6IDIwLjAKICBjYW1lcmE6CiAgICBpc2FhYzogeyBwcmltX3BhdGg6ICIiIH0KICAgIG1vZGVsX25hbWU6ICJJbnRlbCBSZWFsU2Vuc2UgRDQzNWkiCiAgICBjb3N0X3VzZDogMjAwCiAgICByYW5nZV9tOiAxMC4wCiAgICBob3Jpem9udGFsX2Zvdl9kZWc6IDg3LjAKICAgIHZlcnRpY2FsX2Zvdl9kZWc6IDU4LjAKICAgIG1hc3Nfa2c6IDAuMDcKICAgIGNvbXB1dGVfdG9wczogMi4wCiAgICBtZW1vcnlfZ2I6IDAuNAogICAgbGF0ZW5jeV9tczogOC4wCiAgcmFkYXI6CiAgICBpc2FhYzogeyBwcmltX3BhdGg6ICIiIH0KICAgIG1vZGVsX25hbWU6ICJUSSBBV1IxODQzIgogICAgY29zdF91c2Q6IDY1MAogICAgcmFuZ2VfbTogMzAuMAogICAgaG9yaXpvbnRhbF9mb3ZfZGVnOiAxMjAuMAogICAgdmVydGljYWxfZm92X2RlZzogMzAuMAogICAgbWFzc19rZzogMC4wNQogICAgY29tcHV0ZV90b3BzOiAxLjAKICAgIG1lbW9yeV9nYjogMC4yCiAgICBsYXRlbmN5X21zOiA2LjAKCm1vdW50aW5nX3Nsb3RzOgogIC0gcHJpc21fZnJvbnRfZmFjZV9sCiAgLSBwcmlzbV9mcm9udF9mYWNlX3IKICAtIHByaXNtX3RvcF9lZGdlX2wKICAtIHByaXNtX3RvcF9lZGdlX3IKICAtIHByaXNtX2xlZnRfZWRnZQogIC0gcHJpc21fcmlnaHRfZWRnZQoKY21hOgogIHNpZ21hMDogMC4zNQogIHBvcHVsYXRpb25fc2l6ZTogNAogIG1heF9nZW5lcmF0aW9uczogMgogIHRvbHg6IDFlLTkKICB0b2xmdW46IDFlLTkKCmxvc3M6CiAgbW9kZTogZGVmYXVsdAogIGFscGhhOiAwLjQKICBiZXRhOiAwLjQKICBnYW1tYTogMC4yCiAgbWF4X2Nvc3RfdXNkOiAxMDAwMC4wCgppbm5lcl9sb29wOgogIG1vZGU6ICJtdWpvY28iCiAgbl9lcGlzb2RlczogMgogIG1heF9zdGVwc19wZXJfZXBpc29kZTogODAKICBpc2FhY19zaW06CiAgICBzZW5zb3Jfbm9pc2Vfc3RkOiAwLjAKICBtdWpvY286CiAgICBudW1fZW52czogMQogICAgc2Vuc29yX25vaXNlX3N0ZDogMC4wCiAgICBwcmlzbV9oYWxmX2V4dGVudHM6IFswLjQsIDAuMiwgMC4xMl0KICAgIGdvYWxfejogMC4yCgpoYXJkd2FyZToKICBuYW1lOiAiamV0c29uX29yaW5fbGlrZSIKICBjb21wdXRlX2xpbWl0X3RvcHM6IDIwLjAKICBtZW1vcnlfbGltaXRfZ2I6IDQuMAogIGxhdGVuY3lfYnVkZ2V0X21zOiA2MC4wCgptdWx0aV9maWRlbGl0eToKICBlbmFibGVkOiBmYWxzZQoKbG9nZ2luZzoKICBjc3Y6IHRydWUKICBtbGZsb3c6IGZhbHNlCiAgcmVzdWx0c19kaXI6ICJyZXN1bHRzIgo="

def _patch_experiment_name(text: str) -> str:
    return text.replace('  name: "mujoco_local_smoke"', '  name: "mujoco_colab_smoke"', 1)

cfg_path = Path("configs/mujoco_colab_smoke.yaml")
cfg_path.parent.mkdir(parents=True, exist_ok=True)
text = base64.b64decode(_MUJOCO_LOCAL_SMOKE_YAML_B64).decode("utf-8")
# Prefer notebook-embedded snapshot over the zip’s configs/ (same as repo `mujoco_local_smoke`)
cfg_path.write_text(_patch_experiment_name(text), encoding="utf-8")
print("Wrote", cfg_path.resolve())
importlib.import_module("sensor_opt")
print("OK: sensor_opt import")


In [ ]:
# --- MuJoCo sanity: minimal model + one step ---
import numpy as np
import mujoco
from mujoco import MjData, MjModel

_xml = r"""
<mujoco model="pendulum">
  <option timestep="0.01"/>
  <worldbody>
    <body>
      <joint type="hinge" axis="0 1 0"/>
      <geom type="capsule" fromto="0 0 0 0.1 0 0" size="0.02"/>
    </body>
  </worldbody>
</mujoco>
""".strip()

m = MjModel.from_xml_string(_xml)
d = MjData(m)
mujoco.mj_step(m, d)
print("qpos after one step:", np.asarray(d.qpos))
print("MuJoCo OK (minimal MJCF + mj_step).")


In [ ]:
# --- CMA-ES + MuJoCo inner loop (smoke) ---
import subprocess
import sys
from pathlib import Path

assert Path("sensor_opt", "__init__.py").is_file(), "Run the materialize + config cells first."

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "sensor_opt.run_experiment",
        "--config",
        "configs/mujoco_colab_smoke.yaml",
        "--no-mlflow",
    ]
)
print("Done. See results/ for generations.csv and final_result.json")


## CMA-ES performance plots


In [ ]:
# Matplotlib: loss curves
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
from sensor_opt.plotting.cma_matplotlib import plot_cma_generations_matplotlib

mpl.rcParams["figure.figsize"] = (8, 4)
mpl.rcParams["axes.grid"] = True
mpl.rcParams["lines.linewidth"] = 2


def resolve_generations_csv(results_dir: str = "results", explicit=None):
    if explicit:
        p = Path(explicit)
        if p.is_file():
            return p
        raise FileNotFoundError(str(p))
    root = Path(results_dir)
    cands = sorted(
        root.glob("*/generations.csv"),
        key=lambda q: q.stat().st_mtime,
        reverse=True,
    )
    if not cands:
        raise FileNotFoundError("No generations.csv — run the experiment cell first (%s)" % root.resolve())
    return cands[0]


RESULTS_CSV = None
csv_path = resolve_generations_csv(explicit=RESULTS_CSV)
print("Using:", csv_path.resolve())
fig = plot_cma_generations_matplotlib(csv_path, title=csv_path.parent.name)
plt.show()
